In [9]:
import os
import pandas as pd
from fastparquet import write
from fastparquet import ParquetFile
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold
from pathlib import Path
import numpy as np
import glob
import os

In [10]:
data_pipeline = "target_encode"
input_pipeline = "baseline"

In [11]:
if os.environ.get('KAGGLE_KERNEL_RUN_TYPE'):
    data_path = "../../kaggle/input/datasets/abhinavneelam/smartphone-addiction/data/"
    output_path = "/kaggle/working/"
else:
    data_path = "../../data/"
    output_path = "../../"

In [12]:
experiment_path = Path(data_path) / f"{data_pipeline}"
experiment_path.mkdir(parents=True, exist_ok=True)

In [13]:
ss = pd.read_csv("../../data/raw/sample_submission.csv")
target_column = ss.columns[-1]
target_column

'addicted_label'

In [14]:
X = ParquetFile(Path(data_path) / f"{input_pipeline}/train.parq").to_pandas()
X_test = ParquetFile(Path(data_path) / f"{input_pipeline}/test.parq").to_pandas()
y = pd.read_csv(Path(data_path) / 'raw/train.csv')[target_column]

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   age                      662440 non-null  float64 
 1   daily_screen_time_hours  595515 non-null  float64 
 2   social_media_hours       557374 non-null  float64 
 3   gaming_hours             564548 non-null  float64 
 4   work_study_hours         639851 non-null  float64 
 5   sleep_hours              646889 non-null  float64 
 6   notifications_per_day    623785 non-null  float64 
 7   app_opens_per_day        610659 non-null  float64 
 8   weekend_screen_time      579306 non-null  float64 
 9   gender                   662335 non-null  category
 10  stress_level             636221 non-null  float64 
 11  academic_work_impact     647145 non-null  float64 
dtypes: category(1), float64(11)
memory usage: 58.7 MB


In [15]:
kf = StratifiedKFold(n_splits=5, random_state=0, shuffle=True)
TE_cols = [f"{col}_TE" for col in X.columns]
X_te = pd.DataFrame(index=X.index, columns=TE_cols, dtype=float)

for train_index, valid_index in kf.split(X, y):
    X_train, X_valid = X.iloc[train_index], X.iloc[valid_index]
    y_train, y_valid = y.iloc[train_index], y.iloc[valid_index]

    te = TargetEncoder(smooth="auto")
    te.fit(X_train, y_train)
    X_te.iloc[valid_index] = te.transform(X_valid)

te = TargetEncoder(smooth="auto")
te.fit(X, y)
X_test_te = pd.DataFrame(
    te.transform(X_test),
    columns=TE_cols,
    index=X_test.index
)

X_te.info()

<class 'pandas.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   age_TE                      691369 non-null  float64
 1   daily_screen_time_hours_TE  691369 non-null  float64
 2   social_media_hours_TE       691369 non-null  float64
 3   gaming_hours_TE             691369 non-null  float64
 4   work_study_hours_TE         691369 non-null  float64
 5   sleep_hours_TE              691369 non-null  float64
 6   notifications_per_day_TE    691369 non-null  float64
 7   app_opens_per_day_TE        691369 non-null  float64
 8   weekend_screen_time_TE      691369 non-null  float64
 9   gender_TE                   691369 non-null  float64
 10  stress_level_TE             691369 non-null  float64
 11  academic_work_impact_TE     691369 non-null  float64
dtypes: float64(12)
memory usage: 63.3 MB


In [16]:
write(Path(data_path) / Path(data_pipeline) / f"train.parq", X_te)
write(Path(data_path) / Path(data_pipeline)  / f"test.parq", X_test_te)